# Can 96 teacher coordinates preserve useful distillation signal?

**Kaggle: enable Internet, select a GPU, then Save Version > Save & Run All.**
One GPU is used. No input dataset is required for a fresh run.

The teacher is the released CODI GPT-2 checkpoint in **explicit-CoT mode**.
The student is DistilGPT2 with an ordinary dense vocabulary head.
Only the cached teacher supervision changes.

**Default: PILOT.** This runs the real target fitting/audits and five pilot student
arms, then evaluates development questions. It does not open final test predictions.
If the pilot shows a positive full-KD gain, set STAGE to "full" and rerun using the
same saved output. Full runs reuse the pilot and perform all locked seed comparisons.

A negative pilot is a scientific stop, not a software failure. Read pilot_report.json
before changing the common training configuration. Any configuration change makes a
new run identity. A smoke run is separate and supports no scientific conclusions.

The notebook embeds the new helpers and clones an immutable base commit.
It therefore works before you commit or push the new notebook to GitHub.

In [ ]:
# EDIT THIS CELL BEFORE STARTING
STAGE = "pilot"             # "pilot" first; "full" after a successful pilot
SMOKE = False              # True = tiny integration run, no scientific claims
SEEDS = [89, 90, 91]        # Paired training seeds; >=3 for formal claims
RUN_SECONDARY = False      # Add ranks 32/64 and weight-SVD96 (24 vs 15 full fits)
RESUME_ROOT = ""            # Exact saved full_<id> / smoke_<id> directory under /kaggle/input
SESSION_HOURS = 9.0         # Leave time for Kaggle to persist the saved outputs
OUTPUT_ROOT = "/kaggle/working/teacher_code_distillation"

# Execution filters partition full work across sessions WITHOUT changing the experiment.
# Leave None to run every locked arm/seed. Incomplete grids never get completion.json.
ARM_FILTER = None           # e.g. ["sft", "full", "lr96"]; full stage only
SEED_FILTER = None          # e.g. [89]; full stage only

# Common training recipe. Do not retune independently by arm.
STUDENT_EPOCHS = 3
STUDENT_LR = 5e-5
KD_ALPHA = 0.5

In [ ]:
EMBEDDED_FILES = {'src/mech/teacher_code_distillation.py': '"""Offline teacher-code KD: byte-exact caches, causal alignment, losses and inference."""\nfrom __future__ import annotations\nimport hashlib\nimport json\nimport os\nimport random\nimport time\nfrom contextlib import nullcontext\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nVOCAB = 50257\nSTUDENT_REVISION = "2290a62682d06624634c1f46a6ad5be0f47f38aa"\nDATA_REVISION = "3101c7d5072418e28b9008a6636bde82a006892c"\nSVAMP_URL = "https://raw.githubusercontent.com/arkilpatel/SVAMP/78e727689e1c1bebfc4be39c446898e8e10b0518/SVAMP.json"\nSVAMP_SHA = "5be77703a6d891ae476d7c082787ad361392aa02453b132516cdd5f4e7934e3e"\nPRIMARY = ("sft", "full", "lr96", "topk", "sample")\nSECONDARY = ("lr32", "lr64", "svd96")\n\n\n@dataclass\nclass Settings:\n    smoke: bool = False\n    seeds: tuple = (89, 90, 91)\n    secondary: bool = False\n    temperature: float = 2.0\n    alpha: float = 0.5\n    lr: float = 5e-5\n    epochs: int = 3\n    effective_batch: int = 16\n    chunk_tokens: int = 32\n    codec_epochs: int = 6\n    codec_states: int = 32768\n    selection_states: int = 8192\n    max_length: int = 1024\n    generation_cap: int = 256\n    split_seed: int = 20260920\n    codec_seed: int = 89\n    checkpoint_steps: int = 20\n    gradient_batches: int = 16\n    bootstrap_samples: int = 10000\n\n    def checked(self):\n        if self.smoke:\n            self.epochs = self.codec_epochs = 1\n            self.effective_batch = 2\n            self.codec_states = 256\n            self.selection_states = 128\n            self.generation_cap = 16\n            self.gradient_batches = 1\n            self.bootstrap_samples = 100\n            self.seeds = (89,)\n        if not 0 < self.alpha < 1 or self.temperature <= 0:\n            raise ValueError("Require 0<alpha<1 and temperature>0")\n        for name in ("epochs", "effective_batch", "chunk_tokens", "codec_epochs",\n                     "codec_states", "selection_states", "checkpoint_steps",\n                     "generation_cap", "gradient_batches", "bootstrap_samples"):\n            if getattr(self, name) <= 0:\n                raise ValueError(f"{name} must be positive")\n        if self.max_length > 1024 or self.max_length < 32:\n            raise ValueError("GPT-2 context must be between 32 and 1024")\n        if not self.seeds or len(set(self.seeds)) != len(self.seeds):\n            raise ValueError("Provide unique paired training seeds")\n        return self\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()\n\n\ndef file_hash(path):\n    result = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(1024 * 1024), b""):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")\n    os.replace(temporary, path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef save_torch(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    torch.save(value, temporary)\n    os.replace(temporary, path)\n\n\ndef load_torch(path):\n    return torch.load(path, map_location="cpu", weights_only=True)\n\n\nclass SessionBudget:\n    def __init__(self, hours=9):\n        self.deadline = time.monotonic() + hours * 3600\n    def check(self):\n        if time.monotonic() > self.deadline - 180:\n            raise BudgetReached("Session budget reached; saved progress can resume.")\n\n\nclass BudgetReached(RuntimeError):\n    pass\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef rng_state():\n    n = np.random.get_state()\n    return dict(python=random.getstate(), numpy=[n[0], n[1].tolist(), n[2], n[3], n[4]],\n                torch=torch.get_rng_state(),\n                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])\n\n\ndef restore_rng(state):\n    random.setstate(state["python"])\n    n = state["numpy"]\n    np.random.set_state((n[0], np.array(n[1], dtype=np.uint32), n[2], n[3], n[4]))\n    torch.set_rng_state(state["torch"])\n    if state["cuda"] and torch.cuda.is_available():\n        torch.cuda.set_rng_state_all(state["cuda"])\n\n\ndef autocast(device):\n    return torch.autocast("cuda", dtype=torch.float16) if torch.device(device).type == "cuda" else nullcontext()\n\n\ndef question_key(text):\n    return " ".join(text.casefold().split())\n\n\ndef normalize_question(text):\n    return str(text).strip().replace("  ", " ")\n\n\ndef partition_rows(rows, settings):\n    from src.data.answer_extract import normalize_gold\n    unique = {}\n    for row in rows:\n        key = question_key(row["question"])\n        if key in unique and normalize_gold(unique[key]["answer"], "gsm8k_main") != normalize_gold(row["answer"], "gsm8k_main"):\n            raise ValueError("Duplicate question with inconsistent gold answer")\n        unique.setdefault(key, row)\n    ordered = [unique[k] for k in sorted(unique)]\n    random.Random(settings.split_seed).shuffle(ordered)\n    sizes = [8, 4, 4, 8] if settings.smoke else [1024, 256, 256, 512]\n    if len(ordered) <= sum(sizes):\n        raise ValueError("Insufficient unique training questions")\n    result, cursor = {}, 0\n    for name, count in zip(("fit", "select", "audit", "dev"), sizes):\n        result[name] = ordered[cursor:cursor+count]\n        cursor += count\n    result["train"] = ordered[cursor:cursor+16] if settings.smoke else ordered[cursor:]\n    return result\n\n\ndef encode_row(row, tokenizer, max_length=1024):\n    from src.data.answer_extract import normalize_gold\n    gold = normalize_gold(row["answer"], "gsm8k_main")\n    if gold is None or "####" not in row["answer"]:\n        raise ValueError("Missing canonical numeric answer")\n    question = normalize_question(row["question"])\n    rationale = row["answer"].rsplit("####", 1)[0].strip()\n    prompt = tokenizer.encode(question, add_special_tokens=False)\n    reason = tokenizer.encode(" " + rationale, add_special_tokens=False)\n    completion = tokenizer.encode(" " + rationale + " The answer is: " + str(gold),\n                                  add_special_tokens=False) + [tokenizer.eos_token_id]\n    ids = prompt + completion\n    if not prompt or max(ids) >= VOCAB:\n        raise ValueError("Invalid original-vocabulary token sequence")\n    if len(ids) > max_length:\n        return None\n    return dict(id=fingerprint(question_key(question)), question=question, gold=str(gold),\n                ids=ids, prompt_length=len(prompt), count=len(completion),\n                rationale_length=min(len(reason), len(completion)-1))\n\n\ndef labels_and_positions(row):\n    # Position p-1 predicts the first completion token at p; last state predicts EOS.\n    p = row["prompt_length"]\n    positions = np.arange(p-1, len(row["ids"])-1, dtype=np.int64)\n    labels = np.array(row["ids"][p:], dtype=np.int64)\n    if len(positions) != row["count"] or len(labels) != row["count"]:\n        raise ValueError("Corrupt causal token alignment")\n    return labels, positions\n\n\ndef array_write(path, values):\n    path = Path(path)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    np.ascontiguousarray(values).tofile(temp)\n    os.replace(temp, path)\n\n\ndef map_array(path, dtype, shape, create=False):\n    return np.memmap(path, dtype=dtype, mode="w+" if create else "r", shape=tuple(shape))\n\n\ndef soft_cross_entropy(logits, probability, temperature=2.0):\n    return -(probability * F.log_softmax(logits.float()/temperature, -1)).sum(-1)\n\n\ndef sparse_kd(logits, target, temperature=2.0):\n    logq = F.log_softmax(logits.float()/temperature, -1)\n    if target["kind"] == "sample":\n        return -logq.gather(1, target["indices"].long()).mean(-1)\n    indices = target["indices"].long()\n    lp = torch.cat((target["logp"].float(), target["tail"].float().reshape(-1, 1)), -1)\n    p = lp.softmax(-1)\n    selected = logq.gather(1, indices)\n    tail = logq.scatter(1, indices, -torch.inf).logsumexp(-1)\n    return -(p[:, :-1]*selected).sum(-1) - p[:, -1]*tail\n\n\ndef distillation_loss(logits, labels, target=None, alpha=.5, temperature=2.0):\n    ce = F.cross_entropy(logits.float(), labels.long(), reduction="none")\n    if target is None:\n        return ce\n    if target["kind"] == "dense":\n        kd = soft_cross_entropy(logits, target["probability"], temperature)\n    else:\n        kd = sparse_kd(logits, target, temperature)\n    return (1-alpha)*ce + alpha*temperature**2*kd\n\n\ndef exact_budget(n, rank, vocab, metadata_bytes=4096):\n    if n <= 0 or vocab > 65535 or rank <= 0:\n        raise ValueError("Invalid count, rank, or uint16 vocabulary")\n    # Every standalone package has an equal 4096-byte schema; arrays have no headers.\n    lr_bytes = metadata_bytes + 2*n*rank + 2*vocab*(rank+1)\n    available = lr_bytes - metadata_bytes\n    k = min(vocab-1, max(1, (available//n-4)//4))\n    m = max(1, (available//n)//2)\n    return dict(lr_bytes=lr_bytes, k=int(k), m=int(m),\n                topk_bytes=metadata_bytes+n*(4*k+4),\n                sample_bytes=metadata_bytes+2*n*m)\n\n\ndef padded_schema(path, value, size=4096):\n    raw = json.dumps(value, sort_keys=True).encode()\n    if len(raw) > size:\n        raise ValueError("Cache schema exceeds its counted size")\n    Path(path).write_bytes(raw + b" "*(size-len(raw)))\n\n\ndef gradient_comparison(reference, candidate, epsilon=1e-12):\n    r, c = reference.double().reshape(-1), candidate.double().reshape(-1)\n    rn, cn = float(r.norm()), float(c.norm())\n    if rn <= epsilon:\n        return dict(skipped=True)\n    return dict(skipped=False, cosine=float(torch.dot(r,c)/(r.norm()*c.norm().clamp_min(epsilon))),\n                relative_error=float((r-c).norm()/r.norm()), norm_ratio=cn/rn)\n\n\ndef paired_summary(reference, candidate, samples=10000, seed=20260920):\n    """[paired seeds, questions]; bootstrap seeds/questions as crossed factors."""\n    a, b = np.asarray(reference, dtype=float), np.asarray(candidate, dtype=float)\n    if a.ndim != 2 or a.shape != b.shape or a.shape[1] == 0:\n        raise ValueError("Need equal nonempty seed-by-question arrays")\n    delta = b-a\n    generator = np.random.default_rng(seed)\n    observed = float(delta.mean())\n    conditional, crossed = [], []\n    for _ in range(samples):\n        q = generator.integers(a.shape[1], size=a.shape[1])\n        s = generator.integers(a.shape[0], size=a.shape[0])\n        conditional.append(float(delta[:, q].mean()))\n        crossed.append(float(delta[s][:, q].mean()))\n    def ci(x):\n        return (100*np.quantile(x, [.025, .975])).tolist()\n    p = (1+np.count_nonzero(np.asarray(crossed)-observed >= observed))/(samples+1)\n    return dict(delta_pp=100*observed, question_ci95_pp=ci(conditional),\n                crossed_ci95_pp=ci(crossed), p_superiority=float(p),\n                seed_delta_pp=(100*delta.mean(1)).tolist(),\n                changed_correct_to_wrong=((a==1)&(b==0)).sum(1).tolist(),\n                changed_wrong_to_correct=((a==0)&(b==1)).sum(1).tolist(),\n                discordance=float((a!=b).mean()))\n\n\ndef holm(pvalues, alpha=.05):\n    order = np.argsort(pvalues)\n    passed = [False]*len(pvalues)\n    active = True\n    for i, index in enumerate(order):\n        active = active and pvalues[index] <= alpha/(len(order)-i)\n        passed[int(index)] = bool(active)\n    return passed\n\n\n@torch.no_grad()\ndef generate_one(model, tokenizer, question, device, cap=256, vocab=VOCAB):\n    """Plain GPT-2 body path; no logits at prompt positions that are not decoded."""\n    prompt = tokenizer.encode(normalize_question(question), add_special_tokens=False)\n    if len(prompt)+cap > model.config.n_positions:\n        raise ValueError("Evaluation prompt + cap exceeds model context")\n    ids = torch.tensor([prompt], device=device)\n    generated, cache = [], None\n    model.eval()\n    with autocast(device):\n        for _ in range(cap):\n            out = model.transformer(input_ids=ids, past_key_values=cache,\n                                    use_cache=True, return_dict=True)\n            cache = out.past_key_values\n            logits = F.linear(out.last_hidden_state[:, -1], model.get_output_embeddings().weight[:vocab])\n            token = int(logits.argmax(-1))\n            generated.append(token)\n            if token == tokenizer.eos_token_id:\n                break\n            ids = torch.tensor([[token]], device=device)\n    return dict(text=tokenizer.decode(generated, skip_special_tokens=True),\n                tokens=generated, count=len(generated),\n                cap_hit=len(generated)==cap and generated[-1]!=tokenizer.eos_token_id,\n                eos=generated[-1]==tokenizer.eos_token_id)\n\n\ndef evaluate_model(model, tokenizer, rows, output_path, device, budget, cap=256):\n    from src.data.answer_extract import answers_match, extract_final_number\n    output_path = Path(output_path)\n    manifest = dict(ids=[r["id"] for r in rows], cap=cap)\n    meta = output_path.with_suffix(".manifest.json")\n    if meta.exists() and read_json(meta) != manifest:\n        raise ValueError("Evaluation population/cap changed")\n    save_json(meta, manifest)\n    predictions = []\n    if output_path.exists():\n        # Atomic whole-prefix rewrites avoid a partial last line after interruption.\n        predictions = read_json(output_path)\n        if [r["id"] for r in predictions] != manifest["ids"][:len(predictions)]:\n            raise ValueError("Evaluation predictions are not a matching prefix")\n    for row in rows[len(predictions):]:\n        budget.check()\n        start = time.perf_counter()\n        result = generate_one(model, tokenizer, row["question"], device, cap)\n        parsed = extract_final_number(result["text"])\n        result.update(id=row["id"], gold=row["gold"],\n                      parsed=None if parsed is None else str(parsed),\n                      correct=answers_match(result["text"], row["gold"]),\n                      elapsed_seconds=time.perf_counter()-start)\n        predictions.append(result)\n        save_json(output_path, predictions)\n        if len(predictions)%50 == 0:\n            print(f"Evaluation {output_path.stem}: {len(predictions)}/{len(rows)}", flush=True)\n    return predictions\n\n\n', 'scripts/run_teacher_code_distillation.py': '"""Kaggle pipeline for offline teacher-code distillation. No work runs on import."""\nfrom __future__ import annotations\nimport gc\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport shutil\nimport time\nfrom dataclasses import asdict\nfrom functools import partial\nfrom importlib.metadata import version\nfrom pathlib import Path\nfrom urllib.request import urlopen\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\n\nfrom src.mech import teacher_code_distillation as kd\nfrom src.mech.global_low_rank_head import NestedLowRankVocabularyHead, activation_whitened_factors\n\nPRIMARY = kd.PRIMARY\n\n\ndef download(url, path, expected=None):\n    path = Path(path)\n    if not path.exists():\n        with urlopen(url, timeout=300) as response:\n            content = response.read()\n        temp = path.with_suffix(".download")\n        temp.parent.mkdir(parents=True, exist_ok=True)\n        temp.write_bytes(content)\n        os.replace(temp, path)\n    if expected and kd.file_hash(path) != expected:\n        raise ValueError(f"Dataset checksum mismatch: {path}")\n    return path.read_bytes()\n\n\ndef prepare_data(root, tokenizer, settings):\n    from src.data.answer_extract import normalize_gold, normalize_number\n    data = root/"datasets"\n    url = f"https://raw.githubusercontent.com/openai/grade-school-math/{kd.DATA_REVISION}/grade_school_math/data/"\n    train_raw = download(url+"train.jsonl", data/"train.jsonl")\n    test_raw = download(url+"test.jsonl", data/"test.jsonl")\n    svamp_raw = download(kd.SVAMP_URL, data/"SVAMP.json", kd.SVAMP_SHA)\n    raw = [json.loads(x) for x in train_raw.splitlines() if x.strip()]\n    partitions = kd.partition_rows(raw, settings)\n    encoded, excluded = {}, []\n    for split, rows in partitions.items():\n        encoded[split] = []\n        offset = 0\n        for row in rows:\n            item = kd.encode_row(row, tokenizer, settings.max_length)\n            if item is None:\n                excluded.append(dict(split=split, question=row["question"], reason="overlength"))\n                continue\n            item["offset"] = offset\n            offset += item["count"]\n            encoded[split].append(item)\n        if not encoded[split]:\n            raise ValueError(f"Empty eligible split: {split}")\n    tests = {"gsm8k": [], "svamp": []}\n    for line in test_raw.splitlines():\n        row = json.loads(line)\n        tests["gsm8k"].append(dict(question=row["question"],\n                                  gold=str(normalize_gold(row["answer"], "gsm8k_main"))))\n    for row in json.loads(svamp_raw):\n        gold = normalize_number(str(row["Answer"]))\n        if gold is None:\n            raise ValueError("Non-numeric SVAMP answer")\n        tests["svamp"].append(dict(question=row["Body"].strip()+" "+row["Question"].strip(),\n                                 gold=str(gold)))\n    assert len(tests["gsm8k"]) == 1319 and len(tests["svamp"]) == 1000\n    train_keys = {kd.question_key(r["question"]) for rows in partitions.values() for r in rows}\n    for name, rows in tests.items():\n        kept = []\n        for row in rows:\n            row["id"] = kd.fingerprint(kd.question_key(row["question"]))\n            if kd.question_key(row["question"]) in train_keys:\n                excluded.append(dict(split=name, question=row["question"], reason="training overlap"))\n            else:\n                kept.append(row)\n        tests[name] = kept[:8] if settings.smoke else kept\n    manifest = dict(partitions=encoded, tests=tests, excluded=excluded,\n                    data_hashes={p.name: kd.file_hash(p) for p in data.iterdir() if p.is_file()})\n    path = root/"partitions.json"\n    if path.exists() and kd.read_json(path) != manifest:\n        raise ValueError("Tokenized partitions changed")\n    kd.save_json(path, manifest)\n    return encoded, tests\n\n\ndef load_teacher(device):\n    import yaml\n    from src.models.official_codi import (\n        build_official_codi_gpt2, download_official_checkpoint,\n        load_official_checkpoint, official_codi_base_model)\n    cfg = yaml.safe_load(Path("configs/official_codi_gpt2.yaml").read_text())\n    ck = cfg["checkpoint"]\n    path = download_official_checkpoint(repo_id=ck["repo_id"], revision=ck["revision"],\n        filename=ck["filename"], expected_sha256=ck["sha256"])\n    wrapper, tok = build_official_codi_gpt2(base_model=cfg["model"]["base_model"],\n        base_revision=cfg["model"]["base_revision"], dtype=torch.float32, settings=cfg["model"])\n    report = load_official_checkpoint(wrapper, path, expected_sha256=ck["sha256"])\n    wrapper.eval().requires_grad_(False)\n    # Merge on CPU in FP32 before fixed FP16 inference.\n    wrapper.codi = wrapper.codi.merge_and_unload()\n    model = official_codi_base_model(wrapper)\n    model.to(device=device, dtype=torch.float16 if torch.device(device).type=="cuda" else torch.float32)\n    return model, tok, report.to_dict()\n\n\ndef student_tokenizer():\n    from transformers import AutoTokenizer\n    return AutoTokenizer.from_pretrained("distilbert/distilgpt2",\n        revision=kd.STUDENT_REVISION, use_fast=False)\n\n\ndef load_student(device):\n    from transformers import AutoModelForCausalLM\n    model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2",\n        revision=kd.STUDENT_REVISION, torch_dtype=torch.float32)\n    if model.config.vocab_size != kd.VOCAB or model.config.n_layer != 6:\n        raise ValueError("Student architecture changed")\n    model.config.use_cache = False\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    return model.to(device)\n\n\n@torch.no_grad()\ndef teacher_parity(model, tokenizer, row, device):\n    from transformers import SuppressTokensLogitsProcessor, LogitsProcessorList\n    ours = kd.generate_one(model, tokenizer, row["question"], device, cap=8)\n    ids = torch.tensor([tokenizer.encode(kd.normalize_question(row["question"]),\n                                       add_special_tokens=False)], device=device)\n    processors = LogitsProcessorList([SuppressTokensLogitsProcessor(list(range(kd.VOCAB, model.config.vocab_size)))])\n    with kd.autocast(device):\n        out = model.generate(ids, attention_mask=torch.ones_like(ids), do_sample=False,\n            max_new_tokens=8, eos_token_id=tokenizer.eos_token_id,\n            pad_token_id=tokenizer.eos_token_id, logits_processor=processors)\n    if out[0, ids.shape[1]:].tolist() != ours["tokens"]:\n        raise ValueError("Teacher body-only generation parity failed")\n\n\n@torch.no_grad()\ndef collect_states(model, rows, folder, device, budget):\n    folder.mkdir(parents=True, exist_ok=True)\n    n = sum(r["count"] for r in rows)\n    d = model.config.n_embd\n    metadata = dict(n=n, width=d, rows=kd.fingerprint(rows))\n    meta_path = folder/"shape.json"\n    if meta_path.exists() and kd.read_json(meta_path) != metadata:\n        raise ValueError("Hidden-state cache population changed")\n    kd.save_json(meta_path, metadata)\n    path = folder/"hidden.bin"\n    exists = path.exists()\n    if exists and path.stat().st_size != n*d*2:\n        raise ValueError("Truncated hidden-state cache")\n    mmap = np.memmap(path, dtype=np.float16, mode="r+" if exists else "w+", shape=(n,d))\n    progress = folder/"progress.json"\n    done = kd.read_json(progress) if progress.exists() else dict(cursor=0, audit=[])\n    if not exists and done["cursor"]:\n        raise ValueError("Missing partially completed cache")\n    model.eval()\n    for i in range(done["cursor"], len(rows)):\n        budget.check()\n        row = rows[i]\n        ids = torch.tensor([row["ids"]], device=device)\n        _, positions = kd.labels_and_positions(row)\n        with kd.autocast(device):\n            out = model.transformer(input_ids=ids, use_cache=False, return_dict=True)\n        hidden = out.last_hidden_state[0, torch.tensor(positions, device=device)]\n        rounded = hidden.half()\n        mmap[row["offset"]:row["offset"]+row["count"]] = rounded.cpu().numpy()\n        mmap.flush()\n        if len(done["audit"]) < 4:\n            h = rounded[:32].float()\n            raw_w = model.get_output_embeddings().weight\n            canonical = F.linear(h, raw_w[:kd.VOCAB].float())\n            with kd.autocast(device):\n                native = F.linear(hidden[:32], raw_w).float()\n            mass = native.softmax(-1)[:, kd.VOCAB:].sum(-1)\n            done["audit"].append(dict(question=row["id"],\n                max_logit_difference=float((canonical-native[:, :kd.VOCAB]).abs().max()),\n                top1_agreement=float((canonical.argmax(-1)==native[:, :kd.VOCAB].argmax(-1)).float().mean()),\n                excluded_special_mass=float(mass.mean())))\n        done["cursor"] = i+1\n        kd.save_json(progress, done)\n        if (i+1)%100 == 0:\n            print(f"Teacher states {folder.name}: {i+1}/{len(rows)}", flush=True)\n    del mmap\n    kd.save_json(folder/"complete.json", dict(**metadata, file_sha256=kd.file_hash(path)))\n    return n\n\n\ndef hidden_cache(root, split):\n    shape = kd.read_json(root/"states"/split/"shape.json")\n    return kd.map_array(root/"states"/split/"hidden.bin", np.float16, (shape["n"], shape["width"]))\n\n\ndef sample_states(root, split, limit, seed):\n    array = hidden_cache(root, split)\n    indices = np.random.default_rng(seed).choice(len(array), min(limit,len(array)), replace=False)\n    return torch.from_numpy(np.array(array[indices], copy=True)).float()\n\n\ndef construct_head(states, weight, seed=89, initialization="whitened"):\n    ranks = (32,64,96)\n    if min(weight.shape) < 96:\n        raise ValueError("Codec requires width and vocabulary >=96")\n    if initialization == "whitened":\n        centre, down, up, bias, _ = activation_whitened_factors(states, weight, 96, seed=seed)\n    else:\n        centre = states.mean(0)\n        with torch.random.fork_rng(devices=[weight.device.index or 0] if weight.is_cuda else []):\n            torch.manual_seed(seed)\n            u, s, v = torch.svd_lowrank(weight, q=min(112,min(weight.shape)), niter=1)\n        down, up, bias = v[:, :96].T, u[:, :96]*s[:96], F.linear(centre, weight)\n    return NestedLowRankVocabularyHead.from_whitened_factors(centre,down,up,bias,ranks)\n\n\n@torch.no_grad()\ndef codec_metrics(head, states, weight, rank=96, batch=32):\n    total = dict(kl_t2=0., kl_t1=0., agreement=0., margin_error=0., top5_overlap=0.)\n    for start in range(0, len(states), batch):\n        h = states[start:start+batch].to(weight.device).float()\n        z = F.linear(h, weight)\n        # Include actual FP16 code/decoder serialization in selection and diagnostics.\n        c = F.linear(h, head.down.weight[:rank].half().float(), head.down.bias[:rank].half().float()).half().float()\n        zh = F.linear(c, head.up.weight[:, :rank].half().float(), head.up.bias.half().float())\n        for temp, key in ((1,"kl_t1"),(2,"kl_t2")):\n            lp, lq = (z/temp).log_softmax(-1), (zh/temp).log_softmax(-1)\n            total[key] += float((lp.exp()*(lp-lq)).sum())\n        total["agreement"] += float((z.argmax(-1)==zh.argmax(-1)).sum())\n        total["margin_error"] += float(((z.topk(2).values[:,0]-z.topk(2).values[:,1]) -\n                                       (zh.topk(2).values[:,0]-zh.topk(2).values[:,1])).abs().sum())\n        ti, si = z.topk(5).indices, zh.topk(5).indices\n        total["top5_overlap"] += float((ti[:,:,None]==si[:,None,:]).any(-1).float().mean(-1).sum())\n    return {k:v/len(states) for k,v in total.items()}\n\n\ndef fit_codec(root, settings, device, budget, name="whitened"):\n    out = root/"codecs"/name\n    out.mkdir(parents=True, exist_ok=True)\n    final = out/"final.pt"\n    if final.exists():\n        return kd.load_torch(final)\n    w = kd.load_torch(root/"teacher_readout.pt")["weight"].to(device).float()\n    fit = sample_states(root,"fit",settings.codec_states,settings.codec_seed).to(device)\n    select = sample_states(root,"select",settings.selection_states,settings.codec_seed+1)\n    head = construct_head(fit,w,settings.codec_seed,name)\n    optimizer = torch.optim.AdamW(head.parameters(), lr=2e-4, weight_decay=0)\n    resume = out/"resume.pt"\n    cursor, best, best_loss, history = 0, None, math.inf, []\n    if resume.exists():\n        state = kd.load_torch(resume)\n        head.load_state_dict(state["head"])\n        optimizer.load_state_dict(state["optimizer"])\n        cursor,best,best_loss,history = state["epoch"],state["best"],state["best_loss"],state["history"]\n    initial = out/"initial_audit.json"\n    if not initial.exists():\n        kd.save_json(initial, {str(r):codec_metrics(head,select,w,r) for r in (32,64,96)})\n    def snapshot(epoch):\n        kd.save_torch(resume,dict(head=head.state_dict(), optimizer=optimizer.state_dict(),\n                                 epoch=epoch,best=best,best_loss=best_loss,history=history))\n    # Codec interruption resumes the last completed epoch; optimizer checkpoint is explicit.\n    for epoch in range(cursor, settings.codec_epochs):\n        budget.check()\n        order = torch.randperm(len(fit), generator=torch.Generator().manual_seed(settings.codec_seed+epoch))\n        losses = []\n        for start in range(0,len(order),32):\n            if start%1024 == 0:\n                budget.check()\n            h = fit[order[start:start+32]].float()\n            with torch.no_grad():\n                p = (F.linear(h,w)/2).softmax(-1)\n            loss = sum(kd.soft_cross_entropy(head.forward_rank(h,r),p,2).mean()*4\n                       for r in (32,64,96))/3\n            if not torch.isfinite(loss):\n                raise ValueError("Nonfinite codec loss")\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(head.parameters(),1.)\n            optimizer.step()\n            losses.append(float(loss))\n        metrics = codec_metrics(head,select,w)\n        history.append(dict(epoch=epoch+1,training_cross_entropy=float(np.mean(losses)),**metrics))\n        if metrics["kl_t2"] < best_loss:\n            best_loss = metrics["kl_t2"]\n            best = {k:v.detach().cpu().clone() for k,v in head.state_dict().items()}\n        snapshot(epoch+1)\n        print(f"Codec {name}, epoch {epoch+1}: {metrics}",flush=True)\n    head.load_state_dict(best)\n    audit = sample_states(root,"audit",settings.selection_states,settings.codec_seed+2)\n    report = {str(r):codec_metrics(head,audit,w,r) for r in (32,64,96)}\n    exported = {k:v.detach().cpu().half() for k,v in head.state_dict().items()}\n    kd.save_torch(final,exported)\n    kd.save_json(out/"report.json",dict(history=history,audit=report,\n        selected_epoch=1+int(np.argmin([r["kl_t2"] for r in history]))))\n    if resume.exists():\n        resume.unlink()\n    return exported\n\n\n\ndef codec_tensors(state, device, rank=96):\n    # Encoder is also serialized at FP16; no hidden high-precision fitting state.\n    return tuple(state[name].to(device).float() for name in\n                 ("down.weight","down.bias","up.weight","up.bias"))\n\n\ndef build_target_packages(root, settings, device, budget):\n    hidden = hidden_cache(root,"train")\n    n, width = hidden.shape\n    weight = kd.load_torch(root/"teacher_readout.pt")["weight"]\n    vocab = weight.shape[0]\n    costs = kd.exact_budget(n,96,vocab)\n    output = root/"targets"\n    output.mkdir(exist_ok=True)\n    arms = list(PRIMARY[1:]) + (list(kd.SECONDARY) if settings.secondary else [])\n    packages = []\n    for arm in arms:\n        for seed in (settings.seeds if arm=="sample" else (None,)):\n            budget.check()\n            name = f"sample_{seed}" if seed is not None else arm\n            folder = output/name\n            folder.mkdir(exist_ok=True)\n            marker = folder/"complete.json"\n            if marker.exists():\n                packages.append(kd.read_json(marker))\n                continue\n            rank = int(arm[2:]) if arm.startswith("lr") else 96\n            codec = kd.load_torch(root/"codecs"/("weight_svd" if arm=="svd96" else "whitened")/"final.pt")\n            down, db, up, ub = codec_tensors(codec,device,rank)\n            schema = dict(arm=arm, n=n,width=width,vocab=vocab,rank=rank,\n                          k=costs["k"],m=costs["m"],seed=seed,temperature=settings.temperature)\n            kd.padded_schema(folder/"schema.json",schema)\n            definitions = {}\n            if arm=="full":\n                destination = folder/"hidden.bin"\n                if not destination.exists():\n                    try:\n                        os.link(root/"states"/"train"/"hidden.bin", destination)\n                    except OSError:\n                        shutil.copyfile(root/"states"/"train"/"hidden.bin",destination)\n                kd.array_write(folder/"weight.bin",weight.half().numpy())\n            elif arm.startswith("lr") or arm=="svd96":\n                kd.array_write(folder/"up.bin",up[:, :rank].half().cpu().numpy())\n                kd.array_write(folder/"bias.bin",ub.half().cpu().numpy())\n                definitions = {"codes": (np.float16,(n,rank))}\n            elif arm=="topk":\n                definitions = {"indices":(np.uint16,(n,costs["k"])),\n                               "logp":(np.float16,(n,costs["k"])),\n                               "tail":(np.float32,(n,))}\n            else:\n                definitions = {"indices":(np.uint16,(n,costs["m"]))}\n            arrays = {}\n            for key,(dtype,shape) in definitions.items():\n                p = folder/(key+".bin")\n                exists = p.exists()\n                if exists and p.stat().st_size != math.prod(shape)*np.dtype(dtype).itemsize:\n                    raise ValueError(f"Truncated target file {p}")\n                arrays[key] = np.memmap(p,dtype=dtype,mode="r+" if exists else "w+",shape=shape)\n            progress = folder/"progress.json"\n            cursor = kd.read_json(progress)["cursor"] if progress.exists() else 0\n            w = weight.to(device).float()\n            for start in range(cursor,n,256) if definitions else ():\n                budget.check()\n                stop = min(start+256,n)\n                h = torch.from_numpy(np.array(hidden[start:stop],copy=True)).to(device).float()\n                with torch.no_grad():\n                    if arm.startswith("lr") or arm=="svd96":\n                        values = F.linear(h,down[:rank],db[:rank]).half()\n                        arrays["codes"][start:stop] = values.cpu().numpy()\n                    else:\n                        lp = (F.linear(h,w)/settings.temperature).log_softmax(-1)\n                        if arm=="topk":\n                            top,indices = lp.topk(costs["k"],dim=-1)\n                            tail = lp.scatter(1,indices,-torch.inf).logsumexp(-1)\n                            arrays["indices"][start:stop] = indices.cpu().numpy().astype(np.uint16)\n                            arrays["logp"][start:stop] = top.half().cpu().numpy()\n                            arrays["tail"][start:stop] = tail.cpu().numpy()\n                        else:\n                            generator = torch.Generator(device=device).manual_seed(int(seed)*1000003+start)\n                            draws = torch.multinomial(lp.exp(),costs["m"],replacement=True,generator=generator)\n                            arrays["indices"][start:stop] = draws.cpu().numpy().astype(np.uint16)\n                if stop==n or stop%4096==0:\n                    for array in arrays.values():\n                        array.flush()\n                    kd.save_json(progress,dict(cursor=stop))\n            for array in arrays.values():\n                array.flush()\n            arrays.clear()\n            files = [folder/"schema.json",*sorted(folder.glob("*.bin"))]\n            measured = sum(p.stat().st_size for p in files)\n            if arm in ("topk","sample") and measured > costs["lr_bytes"]:\n                raise ValueError("Sparse cache exceeds LR96 TOTAL byte budget")\n            report = dict(name=name,arm=arm,seed=seed,total_bytes=measured,\n                token_count=n,payload_budget=costs,\n                files={p.name:dict(bytes=p.stat().st_size,sha256=kd.file_hash(p)) for p in files})\n            kd.save_json(marker,report)\n            packages.append(report)\n            print(f"Target package {name}: {measured/1e6:.2f} MB",flush=True)\n            del w,down,db,up,ub\n    kd.save_json(root/"storage.json",dict(packages=packages,positions=n,\n        materialized_full_logits_bytes=n*vocab*2,\n        note="Research cache/archive is additional. Each sample seed is one separately deployed cache."))\n    return packages\n\n\nclass TargetReader:\n    def __init__(self,root,arm,seed,device):\n        self.device = device\n        self.arm = arm\n        self.arrays = {}\n        if arm=="sft":\n            return\n        folder = root/"targets"/(f"sample_{seed}" if arm=="sample" else arm)\n        if not (folder/"complete.json").exists():\n            raise ValueError("Target package is incomplete")\n        self.schema = kd.read_json(folder/"schema.json")\n        s = self.schema\n        self.temperature = s["temperature"]\n        def read(name,dtype,shape):\n            return kd.map_array(folder/(name+".bin"),dtype,shape)\n        if arm=="full":\n            self.arrays["hidden"] = read("hidden",np.float16,(s["n"],s["width"]))\n            self.weight = torch.from_numpy(np.array(read("weight",np.float16,(s["vocab"],s["width"])),copy=True)).to(device).float()\n        elif arm.startswith("lr") or arm=="svd96":\n            self.arrays["codes"] = read("codes",np.float16,(s["n"],s["rank"]))\n            self.up = torch.from_numpy(np.array(read("up",np.float16,(s["vocab"],s["rank"])),copy=True)).to(device).float()\n            self.bias = torch.from_numpy(np.array(read("bias",np.float16,(s["vocab"],)),copy=True)).to(device).float()\n        else:\n            count = s["k"] if arm=="topk" else s["m"]\n            self.arrays["indices"] = read("indices",np.uint16,(s["n"],count))\n            if arm=="topk":\n                self.arrays["logp"] = read("logp",np.float16,(s["n"],count))\n                self.arrays["tail"] = read("tail",np.float32,(s["n"],))\n\n    @torch.no_grad()\n    def get(self,start,stop):\n        if self.arm=="sft":\n            return None\n        values = {name:torch.from_numpy(np.array(a[start:stop],copy=True).astype(\n            np.int64 if name=="indices" else np.float32)).to(self.device)\n                  for name,a in self.arrays.items()}\n        with torch.autocast(torch.device(self.device).type,enabled=False):\n            if self.arm=="full":\n                return dict(kind="dense",probability=(F.linear(values["hidden"],self.weight)/self.temperature).softmax(-1))\n            if self.arm.startswith("lr") or self.arm=="svd96":\n                return dict(kind="dense",probability=(F.linear(values["codes"],self.up,self.bias)/self.temperature).softmax(-1))\n        return dict(kind="topk" if self.arm=="topk" else "sample",**values)\n\n\ndef epoch_batches(rows,seed,epoch,batch):\n    order = list(range(len(rows)))\n    random.Random(seed*100003+epoch).shuffle(order)\n    # The same shuffled chunks and order are shared by all arms.\n    return [order[i:i+batch] for i in range(0,len(order),batch)]\n\n\ndef train_student(model,rows,target,folder,settings,seed,device,budget):\n    """Optimizer-step atomic resume, paired RNG, chunked checkpointed vocabulary loss."""\n    folder.mkdir(parents=True,exist_ok=True)\n    done = folder/"final.pt"\n    identity = dict(rows=kd.fingerprint(rows),seed=seed,settings=asdict(settings),arm=target.arm)\n    meta = folder/"train_manifest.json"\n    if meta.exists() and kd.read_json(meta) != json.loads(json.dumps(identity)):\n        raise ValueError("Student training identity changed")\n    kd.save_json(meta,identity)\n    if done.exists() and (folder/"training.json").exists():\n        model.load_state_dict(kd.load_torch(done))\n        return kd.read_json(folder/"training.json")\n    decay,no_decay = [],[]\n    for name,param in model.named_parameters():\n        (no_decay if param.ndim<2 or name.endswith("bias") else decay).append(param)\n    optimizer = torch.optim.AdamW([dict(params=decay,weight_decay=.01),\n                                  dict(params=no_decay,weight_decay=0.)],\n                                 lr=settings.lr,betas=(.9,.999),eps=1e-8)\n    scaler = torch.amp.GradScaler("cuda",enabled=torch.device(device).type=="cuda")\n    batches_per_epoch = math.ceil(len(rows)/settings.effective_batch)\n    total_steps = batches_per_epoch*settings.epochs\n    step,history,elapsed = 0,[],0.\n    resume = folder/"resume.pt"\n    kd.seed_all(seed)\n    if resume.exists():\n        state = kd.load_torch(resume)\n        model.load_state_dict(state["model"])\n        optimizer.load_state_dict(state["optimizer"])\n        scaler.load_state_dict(state["scaler"])\n        step,history,elapsed = state["step"],state["history"],state["elapsed"]\n        kd.restore_rng(state["rng"])\n    def snapshot():\n        kd.save_torch(resume,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),\n            scaler=scaler.state_dict(),step=step,history=history,elapsed=elapsed,rng=kd.rng_state()))\n    if torch.device(device).type=="cuda":\n        torch.cuda.reset_peak_memory_stats()\n    model.train()\n    while step<total_steps:\n        try:\n            budget.check()\n        except kd.BudgetReached:\n            snapshot()\n            raise\n        epoch,index = divmod(step,batches_per_epoch)\n        batch_indices = epoch_batches(rows,seed,epoch,settings.effective_batch)[index]\n        denominator = sum(rows[i]["count"] for i in batch_indices)\n        warmup = max(1,math.ceil(.05*total_steps))\n        factor = (step+1)/warmup if step<warmup else .5*(1+math.cos(math.pi*(step-warmup)/max(1,total_steps-warmup)))\n        for group in optimizer.param_groups:\n            group["lr"] = settings.lr*factor\n        optimizer.zero_grad(set_to_none=True)\n        started = time.perf_counter()\n        observed_loss = 0.\n        for i in batch_indices:\n            row = rows[i]\n            labels,positions = kd.labels_and_positions(row)\n            ids = torch.tensor([row["ids"]],device=device)\n            with kd.autocast(device):\n                h = model.transformer(input_ids=ids,use_cache=False,return_dict=True).last_hidden_state[0]\n                h = h[torch.tensor(positions,device=device)]\n                losses = []\n                for start in range(0,len(h),settings.chunk_tokens):\n                    stop = min(start+settings.chunk_tokens,len(h))\n                    label_tensor = torch.tensor(labels[start:stop],device=device)\n                    # Bind indices explicitly: checkpoint recomputation must not capture the final loop slice.\n                    def chunk_loss(hidden,labels,offset,stop_offset):\n                        logits = model.get_output_embeddings()(hidden)\n                        teacher = target.get(offset,stop_offset)\n                        return kd.distillation_loss(logits,labels,teacher,settings.alpha,\n                                                   settings.temperature).sum()/denominator\n                    fn = partial(chunk_loss,offset=row["offset"]+start,stop_offset=row["offset"]+stop)\n                    losses.append(checkpoint(fn,h[start:stop],label_tensor,use_reentrant=False))\n                loss = sum(losses)\n            if not torch.isfinite(loss):\n                raise ValueError("Nonfinite student loss")\n            observed_loss += float(loss.detach())\n            scaler.scale(loss).backward()\n        scaler.unscale_(optimizer)\n        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n        if not torch.isfinite(grad_norm):\n            # Stop; do not silently skip different optimizer updates in different arms.\n            raise ValueError("Nonfinite gradient; lower common pilot LR or revise precision")\n        scaler.step(optimizer)\n        scaler.update()\n        if torch.device(device).type=="cuda":\n            torch.cuda.synchronize()\n        duration = time.perf_counter()-started\n        elapsed += duration\n        step += 1\n        history.append(dict(step=step,epoch=epoch+1,loss=observed_loss,tokens=denominator,\n                            seconds=duration,lr=optimizer.param_groups[0]["lr"]))\n        if step%10==0 or step==total_steps:\n            print(f"{folder.name}: step {step}/{total_steps}, loss {observed_loss:.4f}",flush=True)\n        if step%settings.checkpoint_steps==0:\n            snapshot()\n    kd.save_torch(done,{k:v.detach().cpu().half() if v.is_floating_point() else v.detach().cpu()\n                        for k,v in model.state_dict().items()})\n    metrics = dict(history=history,training_seconds=elapsed,\n        tokens_per_second=sum(x["tokens"] for x in history)/max(elapsed,1e-12),\n        peak_allocated_bytes=torch.cuda.max_memory_allocated() if torch.device(device).type=="cuda" else None,\n        peak_reserved_bytes=torch.cuda.max_memory_reserved() if torch.device(device).type=="cuda" else None)\n    kd.save_json(folder/"training.json",metrics)\n    if resume.exists():\n        resume.unlink()\n    # Evaluate the serialized deployment artifact, identically to a resumed run.\n    model.load_state_dict(kd.load_torch(done))\n    return metrics\n\n\n\n@torch.no_grad()\ndef development_nll(model,rows,folder,settings,device,budget):\n    path=folder/"development_nll.json"\n    records=kd.read_json(path) if path.exists() else []\n    if [r["id"] for r in records] != [r["id"] for r in rows[:len(records)]]:\n        raise ValueError("Development NLL population changed")\n    model.eval()\n    for row in rows[len(records):]:\n        budget.check()\n        labels,positions=kd.labels_and_positions(row)\n        ids=torch.tensor([row["ids"]],device=device)\n        total=0.\n        with kd.autocast(device):\n            hidden=model.transformer(input_ids=ids,use_cache=False,return_dict=True).last_hidden_state[0]\n            for start in range(0,len(positions),settings.chunk_tokens):\n                selected=positions[start:start+settings.chunk_tokens]\n                logits=model.get_output_embeddings()(hidden[torch.tensor(selected,device=device)])\n                y=torch.tensor(labels[start:start+settings.chunk_tokens],device=device)\n                total += float(F.cross_entropy(logits.float(),y,reduction="sum"))\n        records.append(dict(id=row["id"],tokens=row["count"],negative_log_likelihood=total))\n        kd.save_json(path,records)\n    return sum(r["negative_log_likelihood"] for r in records)/sum(r["tokens"] for r in records)\n\ndef prediction_summary(predictions):\n    return dict(n=len(predictions),accuracy=float(np.mean([r["correct"] for r in predictions])),\n                mean_tokens=float(np.mean([r["count"] for r in predictions])),\n                cap_hits=sum(r["cap_hit"] for r in predictions),\n                malformed=sum(r["parsed"] is None for r in predictions))\n\n\n\n@torch.no_grad()\ndef target_on_hidden(h,arm,weight,codec,costs,seed=89,temperature=2.):\n    if arm=="full":\n        return dict(kind="dense",probability=(F.linear(h.float(),weight.float())/temperature).softmax(-1))\n    if arm.startswith("lr") or arm=="svd96":\n        rank = int(arm[2:]) if arm.startswith("lr") else 96\n        down,db,up,ub = codec_tensors(codec,h.device,rank)\n        code = F.linear(h.float(),down[:rank],db[:rank]).half().float()\n        return dict(kind="dense",probability=(F.linear(code,up[:,:rank],ub)/temperature).softmax(-1))\n    lp = (F.linear(h.float(),weight.float())/temperature).log_softmax(-1)\n    if arm=="topk":\n        top,idx = lp.topk(costs["k"],-1)\n        tail = lp.scatter(1,idx,-torch.inf).logsumexp(-1)\n        return dict(kind="topk",indices=idx,logp=top.half().float(),tail=tail)\n    generator = torch.Generator(device=h.device).manual_seed(seed)\n    idx = torch.multinomial(lp.exp(),costs["m"],replacement=True,generator=generator)\n    return dict(kind="sample",indices=idx)\n\n\ndef tuple_gradient_metrics(reference,candidate):\n    rr=cc=rc=ee=0.\n    for r,c in zip(reference,candidate):\n        if r is None and c is None:\n            continue\n        if r is None:\n            r = torch.zeros_like(c)\n        if c is None:\n            c = torch.zeros_like(r)\n        r,c = r.detach().float(),c.detach().float()\n        rr += float(r.square().sum())\n        cc += float(c.square().sum())\n        rc += float((r*c).sum())\n        ee += float((r-c).square().sum())\n    if rr<=1e-12:\n        return dict(skipped=True)\n    return dict(skipped=False,cosine=rc/max(math.sqrt(rr*cc),1e-12),\n                relative_error=math.sqrt(ee/rr),norm_ratio=math.sqrt(cc/rr))\n\n\ndef gradient_audit(root,model,rows,settings,device,budget,label):\n    path = root/f"gradient_audit_{label}.json"\n    records = kd.read_json(path) if path.exists() else []\n    probe_rows = rows[:settings.gradient_batches]\n    weight = kd.load_torch(root/"teacher_readout.pt")["weight"].to(device).float()\n    codec = kd.load_torch(root/"codecs"/"whitened"/"final.pt")\n    hidden = hidden_cache(root,"audit")\n    n = kd.read_json(root/"states"/"train"/"shape.json")["n"]\n    costs = kd.exact_budget(n,96,weight.shape[0])\n    parameters = tuple(p for p in model.parameters() if p.requires_grad)\n    model.eval()\n    for ri,row in enumerate(probe_rows):\n        if any(r["id"]==row["id"] for r in records):\n            continue\n        budget.check()\n        labels,positions = kd.labels_and_positions(row)\n        chosen = np.unique(np.linspace(0,row["count"]-1,min(row["count"],32)).astype(int))\n        teacher_h = torch.from_numpy(np.array(hidden[row["offset"]+chosen],copy=True)).to(device).float()\n        y = torch.tensor(labels[chosen],device=device)\n        ids = torch.tensor([row["ids"]],device=device)\n        with kd.autocast(device):\n            body = model.transformer(input_ids=ids,use_cache=False,return_dict=True).last_hidden_state[0]\n            logits = model.get_output_embeddings()(body[torch.tensor(positions[chosen],device=device)]).float()\n        ce = F.cross_entropy(logits,y)\n        ceg = torch.autograd.grad(ce,parameters,retain_graph=True,allow_unused=True)\n        targets = {arm:target_on_hidden(teacher_h,arm,weight,codec,costs,\n                   settings.codec_seed+ri,settings.temperature) for arm in ("full","lr96","lr32","lr64","topk","sample")}\n        def kd_loss(target, logits=logits):\n            if target["kind"]=="dense":\n                return kd.soft_cross_entropy(logits,target["probability"],settings.temperature).mean()*settings.temperature**2\n            return kd.sparse_kd(logits,target,settings.temperature).mean()*settings.temperature**2\n        full_loss = kd_loss(targets["full"])\n        fg = torch.autograd.grad(full_loss,parameters,retain_graph=True,allow_unused=True)\n        fl = torch.autograd.grad(full_loss,logits,retain_graph=True)[0]\n        cl = torch.autograd.grad(ce,logits,retain_graph=True)[0]\n        result = dict(id=row["id"],positions=chosen.tolist(),arms={})\n        for arm in ("lr96","lr32","lr64","topk","sample"):\n            loss = kd_loss(targets[arm])\n            cg = torch.autograd.grad(loss,parameters,retain_graph=True,allow_unused=True)\n            gl = torch.autograd.grad(loss,logits,retain_graph=True)[0]\n            def combine(gs, ceg=ceg):\n                return tuple(None if a is None and b is None else\n                    (1-settings.alpha)*(torch.zeros_like(b) if a is None else a)\n                    + settings.alpha*(torch.zeros_like(a) if b is None else b)\n                    for a,b in zip(ceg,gs))\n            result["arms"][arm] = dict(\n                parameter_kd=tuple_gradient_metrics(fg,cg),\n                parameter_combined=tuple_gradient_metrics(combine(fg),combine(cg)),\n                logits_kd=kd.gradient_comparison(fl,gl),\n                logits_combined=kd.gradient_comparison((1-settings.alpha)*cl+settings.alpha*fl,\n                                                       (1-settings.alpha)*cl+settings.alpha*gl))\n            del cg\n        records.append(result)\n        kd.save_json(path,records)\n        del ceg,fg,full_loss,logits,body\n        gc.collect()\n    return records\n\n\ndef target_strata_audit(root,rows,settings,device,budget):\n    path=root/"target_strata.json"\n    if path.exists():\n        return kd.read_json(path)\n    h = hidden_cache(root,"audit")\n    w = kd.load_torch(root/"teacher_readout.pt")["weight"].to(device).float()\n    codec=kd.load_torch(root/"codecs"/"whitened"/"final.pt")\n    costs=kd.exact_budget(kd.read_json(root/"states"/"train"/"shape.json")["n"],96,len(w))\n    records=[]\n    for row in rows:\n        budget.check()\n        labels,_=kd.labels_and_positions(row)\n        for start in range(0,row["count"],32):\n            chosen=np.arange(start,min(start+32,row["count"]))\n            hh=torch.from_numpy(np.array(h[row["offset"]+chosen],copy=True)).to(device).float()\n            yy=torch.tensor(labels[chosen],device=device)\n            with torch.no_grad():\n                z=F.linear(hh,w)\n                lp=(z/settings.temperature).log_softmax(-1)\n                p=lp.exp()\n                for rank in (32,64,96):\n                    t=target_on_hidden(hh,f"lr{rank}",w,codec,costs,temperature=settings.temperature)\n                    lq=t["probability"].clamp_min(1e-30).log()\n                    kl=(p*(lp-lq)).sum(-1).cpu().tolist()\n                    agree=(z.argmax(-1)==lq.argmax(-1)).cpu().tolist()\n                    gold_delta=(lp.gather(1,yy[:,None])-lq.gather(1,yy[:,None])).flatten().cpu().tolist()\n                    margin=z.topk(2).values.diff(dim=-1).neg().flatten().cpu().tolist()\n                    for j,pos in enumerate(chosen):\n                        records.append(dict(id=row["id"],position=int(pos),rank=rank,kl_t2=kl[j],\n                            teacher_agreement=agree[j],gold_nll_delta_t2=gold_delta[j],teacher_margin=margin[j],\n                            teacher_matches_gold=bool(z[j].argmax()==yy[j]),\n                            token_type="eos" if pos==row["count"]-1 else\n                                       ("rationale" if pos<row["rationale_length"] else "answer")))\n    kd.save_json(path,records)\n    return records\n\n\ndef summarize_full(root,settings,arms,tests):\n    result=dict(smoke=settings.smoke,seeds=list(settings.seeds),datasets={},gates={})\n    for dataset,rows in tests.items():\n        flags={}\n        table={}\n        for arm in arms:\n            predictions=[kd.read_json(root/"students"/"full"/f"{arm}_{s}"/f"{dataset}.json")\n                         for s in settings.seeds]\n            expected=[r["id"] for r in rows]\n            if any([r["id"] for r in p]!=expected for p in predictions):\n                raise ValueError("Cannot pair predictions with different questions")\n            flags[arm]=np.array([[r["correct"] for r in p] for p in predictions])\n            table[arm]=[prediction_summary(p) for p in predictions]\n        comparisons={}\n        for reference,candidate in (("sft","full"),("full","lr96"),("sft","lr96"),("topk","lr96"),("sample","lr96")):\n            comparisons[f"{candidate}_minus_{reference}"]=kd.paired_summary(\n                flags[reference],flags[candidate],settings.bootstrap_samples,settings.split_seed)\n        result["datasets"][dataset]=dict(arms=table,comparisons=comparisons)\n    comp=result["datasets"]["gsm8k"]["comparisons"]\n    utility=comp["full_minus_sft"]\n    sufficient=comp["lr96_minus_full"]\n    compact=[comp["lr96_minus_topk"],comp["lr96_minus_sample"]]\n    corrected=kd.holm([r["p_superiority"] for r in compact])\n    valid=not settings.smoke and len(settings.seeds)>=3\n    result["gates"]=dict(\n        eligible_for_claims=valid,\n        teacher_utility=valid and utility["delta_pp"]>=1 and utility["crossed_ci95_pp"][0]>0,\n        lr96_noninferiority=valid and sufficient["crossed_ci95_pp"][0]>-1,\n        compact_superiority=valid and all(corrected) and all(r["delta_pp"]>=.5 for r in compact))\n    result["gates"]["overall_pass"]=all(result["gates"].values())\n    result["limitations"]=[\n        "Previously inspected benchmarks; this is a locked follow-up, not a pristine holdout.",\n        "Three seeds give weak seed-population uncertainty; report individual seeds and both intervals.",\n        "Full and hidden-state KD are the same canonical target, not independent experimental arms.",\n        "Explicit reasoning only. CODI latent/student and cross-family extensions are not run here."]\n    kd.save_json(root/"results.json",result)\n    kd.save_json(root/"completion.json",dict(stage="full",smoke=settings.smoke,\n        expected_arms=list(arms),expected_seeds=list(settings.seeds),all_evaluations_complete=True))\n    return result\n\n\ndef run_experiment(output_root,settings=None,stage="pilot",resume_root="",hours=9,\n                   source_hash="",arm_filter=None,seed_filter=None,_device=None):\n    settings=(settings or kd.Settings()).checked()\n    if stage not in ("pilot","full"):\n        raise ValueError("STAGE must be pilot or full")\n    if _device is None and not torch.cuda.is_available():\n        raise RuntimeError("Enable a Kaggle GPU; local tests use tiny model functions directly.")\n    device=torch.device(_device or "cuda:0")\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n    torch.use_deterministic_algorithms(True)\n    identity=dict(settings=asdict(settings),source_hash=source_hash,\n        student_revision=kd.STUDENT_REVISION,data_revision=kd.DATA_REVISION,\n        packages={p:version(p) for p in ("torch","transformers","peft","numpy","huggingface-hub")},\n        gpu=torch.cuda.get_device_name(0) if device.type=="cuda" else "CPU test",cuda=torch.version.cuda,python=platform.python_version(),\n        base_commit="6a8d2e61950c67f012d0a9ba13ec8a70f3a25019")\n    identity=json.loads(json.dumps(identity))\n    run_id=kd.fingerprint(identity)[:16]\n    root=Path(output_root)/(("smoke_" if settings.smoke else "full_")+run_id)\n    # Parentheses matter: construct the path after forming the complete directory name.\n\n\n    if resume_root:\n        source=Path(resume_root)\n        if not (source/"manifest.json").exists() or kd.read_json(source/"manifest.json")!=identity:\n            raise ValueError("RESUME_ROOT must be the exact matching run folder; settings/runtime/source changed.")\n        if not root.exists():\n            print("Restoring saved output to writable storage...",flush=True)\n            shutil.copytree(source,root)\n    root.mkdir(parents=True,exist_ok=True)\n    manifest=root/"manifest.json"\n    if manifest.exists() and kd.read_json(manifest)!=identity:\n        raise ValueError("Existing run has a different identity")\n    kd.save_json(manifest,identity)\n    print(f"Run folder: {root}",flush=True)\n    budget=kd.SessionBudget(hours)\n    started=time.perf_counter()\n    try:\n        tokenizer=student_tokenizer()\n        partitions,tests=prepare_data(root,tokenizer,settings)\n        states_complete=all((root/"states"/s/"complete.json").exists() for s in partitions)\n        if not states_complete or not (root/"teacher_readout.pt").exists():\n            teacher,teacher_tokenizer,load_report=load_teacher(device)\n            if any(tokenizer.convert_ids_to_tokens(i)!=teacher_tokenizer.convert_ids_to_tokens(i)\n                   for i in range(kd.VOCAB)):\n                raise ValueError("Teacher/student token vocabularies disagree")\n            if teacher.config.n_positions<settings.max_length:\n                raise ValueError("Teacher context too short")\n            teacher_parity(teacher,tokenizer,partitions["dev"][0],device)\n            kd.save_json(root/"teacher_load.json",load_report)\n            kd.save_torch(root/"teacher_readout.pt",\n                dict(weight=teacher.get_output_embeddings().weight[:kd.VOCAB].detach().cpu().half()))\n            for split,rows in partitions.items():\n                collect_states(teacher,rows,root/"states"/split,device,budget)\n            del teacher,teacher_tokenizer\n            gc.collect()\n            torch.cuda.empty_cache()\n        for name in ("whitened","weight_svd") if settings.secondary else ("whitened",):\n            fit_codec(root,settings,device,budget,name)\n        report=kd.read_json(root/"codecs"/"whitened"/"report.json")\n        audit=report["audit"]["96"]\n        if not settings.smoke and (audit["agreement"]<.90 or audit["kl_t2"]>.5):\n            result=dict(status="codec_gate_failed",audit=audit,\n                        next_step="Do not scale student training. Review the frozen audit; any redesign is a new protocol.")\n            kd.save_json(root/"gate_result.json",result)\n            return root,result\n        build_target_packages(root,settings,device,budget)\n        target_strata_audit(root,partitions["audit"],settings,device,budget)\n        pilot_rows=partitions["train"][:min(1024,len(partitions["train"]))]\n        pilot_seed=settings.seeds[0]\n        if not (root/"gradient_audit_initial.json").exists() or len(kd.read_json(root/"gradient_audit_initial.json")) < min(settings.gradient_batches,len(partitions["audit"])):\n            model=load_student(device)\n            gradient_audit(root,model,partitions["audit"],settings,device,budget,"initial")\n            del model\n            gc.collect()\n            torch.cuda.empty_cache()\n        pilot_results={}\n        for arm in PRIMARY:\n            budget.check()\n            folder=root/"students"/"pilot"/f"{arm}_{pilot_seed}"\n            model=load_student(device)\n            target=TargetReader(root,arm,pilot_seed,device)\n            train_student(model,pilot_rows,target,folder,settings,pilot_seed,device,budget)\n            predictions=kd.evaluate_model(model,tokenizer,partitions["dev"],folder/"dev.json",\n                                          device,budget,settings.generation_cap)\n            pilot_results[arm]=prediction_summary(predictions)\n            pilot_results[arm]["gold_nll"]=development_nll(model,partitions["dev"],folder,settings,device,budget)\n            if arm=="sft":\n                gradient_audit(root,model,partitions["audit"],settings,device,budget,"sft")\n            del model,target\n            gc.collect()\n            torch.cuda.empty_cache()\n        # Development gate is a feasibility screen, not the final significance gate.\n        gain=100*(pilot_results["full"]["accuracy"]-pilot_results["sft"]["accuracy"])\n        lr_predictions=kd.read_json(root/"students"/"pilot"/f"lr96_{pilot_seed}"/"dev.json")\n        full_predictions=kd.read_json(root/"students"/"pilot"/f"full_{pilot_seed}"/"dev.json")\n        discordance=float(np.mean([a["correct"]!=b["correct"] for a,b in zip(lr_predictions,full_predictions)]))\n        halfwidth=196*math.sqrt(discordance/max(1,len(tests["gsm8k"])))\n        decision=dict(status="pilot_complete",results=pilot_results,full_kd_gain_pp=gain,\n            proceed=bool(settings.smoke or gain>0),development_discordance=discordance,\n            approximate_test_ci_halfwidth_pp=halfwidth,\n            note="Positive pilot gain is only a feasibility screen; final teacher-utility gate needs >=1pp and positive CI.",\n            cache_budget="Sparse controls are matched to the complete full-training cache, not just the pilot subset.")\n        kd.save_json(root/"pilot_report.json",decision)\n        print(json.dumps(decision,indent=2),flush=True)\n        if stage=="pilot":\n            return root,decision\n        if not decision["proceed"]:\n            result=dict(status="pilot_gate_failed",pilot=decision,\n                next_step="Stop: full KD did not improve development accuracy. Review the pilot before revising a COMMON training configuration.")\n            kd.save_json(root/"gate_result.json",result)\n            return root,result\n        lock=dict(settings=asdict(settings),pilot=kd.fingerprint(decision),\n                  tests={k:[r["id"] for r in v] for k,v in tests.items()})\n        lock=json.loads(json.dumps(lock))\n        if (root/"analysis_lock.json").exists() and kd.read_json(root/"analysis_lock.json")!=lock:\n            raise ValueError("The pre-test analysis lock changed")\n        kd.save_json(root/"analysis_lock.json",lock)\n        arms=list(PRIMARY)+(list(kd.SECONDARY) if settings.secondary else [])\n        selected_arms=arms if arm_filter is None else list(arm_filter)\n        selected_seeds=list(settings.seeds) if seed_filter is None else list(seed_filter)\n        if not set(selected_arms)<=set(arms) or not set(selected_seeds)<=set(settings.seeds):\n            raise ValueError("Execution filters must be subsets of the locked arms/seeds")\n        for seed in selected_seeds:\n            for arm in selected_arms:\n                budget.check()\n                folder=root/"students"/"full"/f"{arm}_{seed}"\n                model=load_student(device)\n                target=TargetReader(root,arm,seed,device)\n                train_student(model,partitions["train"],target,folder,settings,seed,device,budget)\n                for dataset,rows in tests.items():\n                    kd.evaluate_model(model,tokenizer,rows,folder/f"{dataset}.json",\n                                      device,budget,settings.generation_cap)\n                del model,target\n                gc.collect()\n                torch.cuda.empty_cache()\n        complete=all((root/"students"/"full"/f"{a}_{s}"/f"{d}.json").exists()\n                     and len(kd.read_json(root/"students"/"full"/f"{a}_{s}"/f"{d}.json"))==len(rows)\n                     for s in settings.seeds for a in arms for d,rows in tests.items())\n        if not complete:\n            return root,dict(status="partial",message="Filtered execution finished. Other locked arms/seeds remain; no completion marker.")\n        return root,summarize_full(root,settings,arms,tests)\n    except kd.BudgetReached as error:\n        result=dict(status="paused_at_session_budget",message=str(error),\n                    instructions="Save Version outputs; attach the saved run and set RESUME_ROOT to its exact directory.")\n        kd.save_json(root/"session_pause.json",result)\n        return root,result\n    finally:\n        path=root/"sessions.json"\n        sessions=kd.read_json(path) if path.exists() else []\n        sessions.append(dict(stage=stage,elapsed_seconds=time.perf_counter()-started))\n        kd.save_json(path,sessions)\n\n\n\n\n\n', 'tests/test_teacher_code_distillation.py': '"""Scientific invariants and tiny CPU end-to-end KD pipeline tests; no downloads."""\nimport copy\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pytest\nimport torch\nfrom transformers import GPT2Config, GPT2LMHeadModel\n\nfrom src.mech import teacher_code_distillation as kd\nfrom scripts import run_teacher_code_distillation as run\n\nROOT = Path(__file__).resolve().parents[1]\n\n\n@pytest.fixture(autouse=True)\ndef deterministic_cpu():\n    torch.set_num_threads(1)\n    torch.manual_seed(19)\n\n\nclass Unlimited:\n    def check(self):\n        pass\n\n\nclass InterruptAfter:\n    def __init__(self,n):\n        self.n=n\n    def check(self):\n        self.n-=1\n        if self.n<0:\n            raise kd.BudgetReached("test interruption")\n\n\ndef tiny_model(vocab=128,width=128):\n    return GPT2LMHeadModel(GPT2Config(vocab_size=vocab,n_embd=width,n_layer=1,n_head=4,\n        n_positions=128,n_ctx=128,bos_token_id=1,eos_token_id=1,pad_token_id=1,\n        resid_pdrop=.1,embd_pdrop=.1,attn_pdrop=.1))\n\n\ndef rows(n=4):\n    result=[]\n    for i in range(n):\n        result.append(dict(id=str(i),question=f"question {i}",gold="1",\n            ids=[2,3+i,4,5,6,1],prompt_length=2,count=4,offset=4*i,rationale_length=2))\n    return result\n\n\ndef test_causal_shift_and_eos():\n    row=rows(1)[0]\n    labels,positions=kd.labels_and_positions(row)\n    assert labels.tolist()==[4,5,6,1]\n    assert positions.tolist()==[1,2,3,4]\n    assert row["ids"][positions[-1]]==6  # State predicts EOS; does not consume it.\n\n\ndef test_topk_tail_is_full_kd_when_only_one_token_is_omitted():\n    logits=torch.randn(4,9,requires_grad=True)\n    lp=torch.randn(4,9).log_softmax(-1)\n    values,indices=lp.topk(8,-1)\n    tail=lp.scatter(1,indices,-torch.inf).logsumexp(-1)\n    full=kd.soft_cross_entropy(logits,lp.exp()).mean()\n    sparse=kd.sparse_kd(logits,dict(kind="topk",indices=indices,logp=values,tail=tail)).mean()\n    assert torch.allclose(full,sparse,atol=1e-6)\n    fg=torch.autograd.grad(full,logits,retain_graph=True)[0]\n    sg=torch.autograd.grad(sparse,logits)[0]\n    assert torch.allclose(fg,sg,atol=1e-6)\n\n\ndef test_sampled_gradient_approximates_dense_gradient():\n    logits=torch.randn(1,12,requires_grad=True)\n    p=torch.randn(1,12).softmax(-1)\n    indices=torch.multinomial(p,200000,replacement=True)\n    full=kd.soft_cross_entropy(logits,p).mean()\n    sampled=kd.sparse_kd(logits,dict(kind="sample",indices=indices)).mean()\n    a=torch.autograd.grad(full,logits,retain_graph=True)[0]\n    b=torch.autograd.grad(sampled,logits)[0]\n    assert torch.allclose(a,b,atol=.002)\n\n\ndef test_exact_budget_includes_decoder_and_uint16_ids():\n    costs=kd.exact_budget(500000,96,50257)\n    assert costs["k"]==51 and costs["m"]==105\n    assert costs["topk_bytes"]<=costs["lr_bytes"]\n    assert costs["sample_bytes"]<=costs["lr_bytes"]\n    assert costs["lr_bytes"]==4096+96000000+9749858\n    with pytest.raises(ValueError):\n        kd.exact_budget(0,96,50257)\n\n\ndef test_paired_statistics_do_not_count_seeds_as_questions():\n    a=np.zeros((3,40),bool); b=a.copy(); b[:,:8]=True\n    report=kd.paired_summary(a,b,samples=200,seed=3)\n    assert report["delta_pp"]==pytest.approx(20)\n    assert report["changed_wrong_to_correct"]==[8,8,8]\n    assert report["crossed_ci95_pp"][0]>0\n    assert kd.holm([.01,.03])==[True,True]\n    assert kd.holm([.04,.06])==[False,False]\n\n\ndef test_training_resumes_exactly_and_chunks_have_correct_labels(tmp_path):\n    original=tiny_model(vocab=32,width=32)\n    cfg=kd.Settings(epochs=2,effective_batch=2,chunk_tokens=2,checkpoint_steps=1)\n    data=rows()\n    class SFT:\n        arm="sft"\n        def get(self,start,stop):\n            return None\n    baseline=copy.deepcopy(original)\n    run.train_student(baseline,data,SFT(),tmp_path/"baseline",cfg,89,"cpu",Unlimited())\n    interrupted=copy.deepcopy(original)\n    with pytest.raises(kd.BudgetReached):\n        run.train_student(interrupted,data,SFT(),tmp_path/"resume",cfg,89,"cpu",InterruptAfter(2))\n    restarted=copy.deepcopy(original)\n    run.train_student(restarted,data,SFT(),tmp_path/"resume",cfg,89,"cpu",Unlimited())\n    for name,value in baseline.state_dict().items():\n        assert torch.equal(value,restarted.state_dict()[name]),name\n    assert not (tmp_path/"resume"/"resume.pt").exists()\n    # Real gradient updates happened.\n    assert not torch.equal(baseline.transformer.wte.weight,original.transformer.wte.weight)\n\n\ndef test_tiny_cache_codec_packages_kd_and_gradient_audit(tmp_path):\n    cfg=kd.Settings(smoke=True,secondary=True).checked()\n    teacher=tiny_model()\n    data=rows(5)\n    for split in ("fit","select","audit","train"):\n        run.collect_states(teacher,data,tmp_path/"states"/split,"cpu",Unlimited())\n    kd.save_torch(tmp_path/"teacher_readout.pt",dict(weight=teacher.lm_head.weight.detach().half()))\n    for name in ("whitened","weight_svd"):\n        run.fit_codec(tmp_path,cfg,"cpu",Unlimited(),name)\n    packages=run.build_target_packages(tmp_path,cfg,"cpu",Unlimited())\n    assert {p["arm"] for p in packages}==set(kd.PRIMARY[1:]+kd.SECONDARY)\n    full=run.TargetReader(tmp_path,"full",89,"cpu")\n    h=run.hidden_cache(tmp_path,"train")[:3]\n    expected=(torch.nn.functional.linear(torch.from_numpy(np.array(h)).float(),\n              teacher.lm_head.weight.detach().half().float())/2).softmax(-1)\n    assert torch.allclose(full.get(0,3)["probability"],expected)\n    for arm in kd.PRIMARY:\n        model=tiny_model()\n        reader=run.TargetReader(tmp_path,arm,89,"cpu")\n        result=run.train_student(model,data,reader,tmp_path/"students"/arm,cfg,89,"cpu",Unlimited())\n        assert len(result["history"])==3\n        assert np.isfinite(result["history"][-1]["loss"])\n    diagnostic=run.gradient_audit(tmp_path,tiny_model(),data,cfg,"cpu",Unlimited(),"tiny")\n    assert len(diagnostic)==1\n    assert "parameter_combined" in diagnostic[0]["arms"]["lr96"]\n    strata=run.target_strata_audit(tmp_path,data,cfg,"cpu",Unlimited())\n    assert {r["token_type"] for r in strata}=={"rationale","answer","eos"}\n\n\ndef test_notebook_embeds_current_source_and_has_no_outputs():\n    path=ROOT/"notebooks/kaggle_teacher_code_distillation.ipynb"\n    notebook=json.loads(path.read_text(encoding="utf-8"))\n    import nbformat\n    nbformat.validate(nbformat.from_dict(notebook))\n    code=["".join(c["source"]) for c in notebook["cells"] if c["cell_type"]=="code"]\n    for i,source in enumerate(code):\n        compile(source,f"cell_{i}","exec")\n    assert all(not c["outputs"] and c["execution_count"] is None\n               for c in notebook["cells"] if c["cell_type"]=="code")\n    # Evaluate only the generated literal definitions; no network/setup execution.\n    ns={}\n    exec(code[1],ns)\n    for relative,source in ns["EMBEDDED_FILES"].items():\n        assert source==(ROOT/relative).read_text(encoding="utf-8-sig")\n    assert "STAGE = \\"pilot\\"" in code[0]\n    assert "run_experiment" in "\\n".join(code)\n\n\n\n\n\ndef test_full_smoke_orchestration_and_filtered_resume(tmp_path,monkeypatch):\n    """Run the production pilot/full orchestration on CPU with no network/model downloads."""\n    class Tokens:\n        eos_token_id=1\n        def encode(self,text,add_special_tokens=False):\n            return [2,3]\n        def decode(self,ids,skip_special_tokens=True):\n            return "1"\n        def convert_ids_to_tokens(self,index):\n            return str(index)\n    tok=Tokens()\n    monkeypatch.setattr(kd,"VOCAB",128)\n    monkeypatch.setattr(run,"student_tokenizer",lambda:tok)\n    monkeypatch.setattr(run,"load_student",lambda device:tiny_model())\n    monkeypatch.setattr(run,"load_teacher",lambda device:(tiny_model().eval(),tok,{"test":True}))\n    monkeypatch.setattr(run,"version",lambda package:"test")\n    data=rows(4)\n    parts={s:copy.deepcopy(data) for s in ("fit","select","audit","dev","train")}\n    tests={"gsm8k":copy.deepcopy(data),"svamp":copy.deepcopy(data)}\n    monkeypatch.setattr(run,"prepare_data",lambda *args:(parts,tests))\n    cfg=kd.Settings(smoke=True,max_length=128)\n    root,result=run.run_experiment(tmp_path,cfg,stage="full",source_hash="test",\n                                  arm_filter=["sft"],_device="cpu")\n    assert result["status"]=="partial"\n    assert (root/"analysis_lock.json").exists()\n    assert not (root/"completion.json").exists()\n    _,result=run.run_experiment(tmp_path,kd.Settings(smoke=True,max_length=128),stage="full",\n                               source_hash="test",_device="cpu")\n    assert (root/"completion.json").exists()\n    assert result["gates"]["eligible_for_claims"] is False\n    assert set(result["datasets"]["gsm8k"]["arms"])==set(kd.PRIMARY)\n\n\n'}
EXPERIMENT_SOURCE_SHA256 = '622c7fddaf548d3accbd22de6f97aba48d9e972edfdec3bc4d0859d02f5515f1'

## Setup and correctness checks

Keep Kaggle's PyTorch/CUDA. Setup installs the compatible Transformers/PEFT stack.
Use a fresh session if unrelated previously imported packages conflict.
All downloads are public; no Hugging Face token is required.

The CPU checks below verify target alignment, sparse-loss gradients, storage matching,
resume parity, and tiny model training. They do not establish GPU speed or math accuracy.

In [ ]:
import importlib
import json
import os
import pathlib
import subprocess
import sys

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
os.environ.setdefault("HF_HOME", "/kaggle/working/teacher_code_hf_cache")

REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
BASE_COMMIT = "6a8d2e61950c67f012d0a9ba13ec8a70f3a25019"
REPO_DIR = pathlib.Path("/kaggle/working/teacher-code-repo")
pathlib.Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

def checked(command):
    result = subprocess.run(command, capture_output=True, text=True)
    with (pathlib.Path(OUTPUT_ROOT)/"setup.log").open("a", encoding="utf-8") as stream:
        stream.write(result.stdout + "\n" + result.stderr + "\n")
    if result.returncode:
        raise RuntimeError(result.stdout[-4000:] + "\n" + result.stderr[-6000:])
    return result.stdout

if not REPO_DIR.exists():
    checked(["git", "clone", REPO_URL, str(REPO_DIR)])
checked(["git", "-C", str(REPO_DIR), "checkout", "--detach", BASE_COMMIT])
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
checked([sys.executable, "-m", "pip", "install", "-q",
         "transformers==4.52.4", "peft==0.15.2", "accelerate==1.7.0",
         "huggingface_hub>=0.34,<1", "hf_xet", "pyyaml", "pytest"])
probe = subprocess.run([sys.executable, "-c", "import peft"], capture_output=True, text=True)
if probe.returncode and "torchao" in (probe.stdout + probe.stderr):
    checked([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
checked([sys.executable, "-c", "import torch, peft; from transformers import GPT2LMHeadModel"])
for relative, source in EMBEDDED_FILES.items():
    destination = REPO_DIR/relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding="utf-8")
# Permit re-import after rerunning cells; old helper modules must not survive an update.
for name in ("scripts.run_teacher_code_distillation", "src.mech.teacher_code_distillation"):
    sys.modules.pop(name, None)
    parent, _, attribute = name.rpartition(".")
    package = sys.modules.get(parent)
    if package is not None and hasattr(package, attribute):
        delattr(package, attribute)
importlib.invalidate_caches()
import torch
assert torch.cuda.is_available(), "Select a GPU in Kaggle's notebook settings."
print("GPU:", torch.cuda.get_device_name(0), "| torch:", torch.__version__)
test_output = checked([sys.executable, "-m", "pytest", "-q",
                       "tests/test_teacher_code_distillation.py", "-k", "not notebook"])
print(test_output)

## Run the selected stage

The same output folder is reused when settings, code, packages, and GPU identity match.
A session budget stop saves a pause report. An unexpected interruption resumes student
training from the last optimizer checkpoint (20 steps by default); teacher collection
resumes by question and target exports by block. Codec fitting resumes completed epochs.

**Across sessions:** save outputs, attach the saved run as Kaggle input, set RESUME_ROOT
to the exact full_<id> directory, and rerun. Restoring copies the run to writable storage.
Do not change seeds, alpha, learning rate, or secondary-arm settings when resuming.

The full grid can exceed one Kaggle session. Use execution filters to split work; all
configured arms and seeds must complete before final gates are reported. Changing only
STAGE, filters, RESUME_ROOT, or SESSION_HOURS does not change the run identity.

In [ ]:
from src.mech import teacher_code_distillation as kd
from scripts.run_teacher_code_distillation import run_experiment

settings = kd.Settings(
    smoke=SMOKE, seeds=tuple(SEEDS), secondary=RUN_SECONDARY,
    epochs=STUDENT_EPOCHS, lr=STUDENT_LR, alpha=KD_ALPHA,
)
RUN_DIR, RESULTS = run_experiment(
    OUTPUT_ROOT, settings=settings, stage=STAGE, resume_root=RESUME_ROOT,
    hours=SESSION_HOURS, source_hash=EXPERIMENT_SOURCE_SHA256,
    arm_filter=ARM_FILTER, seed_filter=SEED_FILTER,
)
print("Saved run:", RUN_DIR)
print(json.dumps(RESULTS, indent=2))

## Read the results

**Pilot:** inspect the codec audit, gradient diagnostics, five development accuracies,
and predicted confidence-interval width. Positive pilot gain only permits the full run;
it does not establish the final utility claim.

**Full:** teacher utility, rank-96 noninferiority, and superiority over both compact
controls are separate gates. Wide intervals mean inconclusive. Smoke runs and fewer
than three seeds are not eligible for claims.

Sparse controls get the same TOTAL byte budget as rank 96, including its shared
reconstruction matrix. Full KD reconstructs exact canonical logits from cached hidden
states; a second mathematically identical full-logit training arm is unnecessary.

In [ ]:
import pandas as pd
from IPython.display import display

if (RUN_DIR/"storage.json").exists():
    storage = kd.read_json(RUN_DIR/"storage.json")
    display(pd.DataFrame([
        dict(package=p["name"], total_MB=p["total_bytes"]/1e6,
             tokens=p["token_count"], bytes_per_token_including_shared=p["total_bytes"]/p["token_count"])
        for p in storage["packages"]
    ]))
if (RUN_DIR/"pilot_report.json").exists():
    pilot = kd.read_json(RUN_DIR/"pilot_report.json")
    display(pd.DataFrame(pilot["results"]).T)
if (RUN_DIR/"results.json").exists():
    result = kd.read_json(RUN_DIR/"results.json")
    for dataset, report in result["datasets"].items():
        print(dataset)
        display(pd.DataFrame([
            dict(arm=arm, seed=seed, **metrics)
            for arm, runs in report["arms"].items()
            for seed, metrics in zip(result["seeds"], runs)
        ]))
        display(pd.DataFrame(report["comparisons"]).T)
    print("Gates:", result["gates"])
else:
    print("No full completion yet:", RESULTS.get("status", "in progress"))

## Preserve outputs

Use **Save Version > Save & Run All** so Kaggle retains the writable output directory.
Download the saved run from Outputs, or attach it to the next session as an input.
Final checkpoints are stored without optimizer state; interrupted fits retain one
optimizer checkpoint. Do not discard that checkpoint when resuming unfinished training.

Files to inspect:
- manifest.json and partitions.json: immutable settings, source identity, splits/exclusions.
- codecs/*/report.json: fitting/selection history and serialized target fidelity.
- target_strata.json and gradient_audit_*.json: token and learning-signal diagnostics.
- storage.json: measured target package sizes, including shared decoder weights.
- pilot_report.json: development outcomes and feasibility decision.
- students/*/*/training.json and dev/gsm8k/svamp.json: training history and every prediction.
- results.json and completion.json: full paired analysis, only after the entire locked grid.
- session_pause.json: normal session-budget stop, with resume instructions.

This notebook implements the primary explicit-CoT experiment and its optional rank/
initializer ablations. The later CODI latent-student extension requires its own experiment.